# Drug-disease prediction scores

💡 **Environment:** `clamp-analyses`

Companion to `00_prediction_performance`: same canonical LV-space `-drug^T . disease`
scores (max-aggregated over the 49 GTEx tissues), shown as a score distribution and
ROC curve (with AUC per method and a chance-level reference) for the three
compendia (ARCHS4, recount2, GTEx) and the gene-space baseline.

In [ ]:
from pathlib import Path
import pandas as pd
import yaml
from pyprojroot import here
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve

with open(here('config.yaml'), 'r') as f:
    colors = yaml.safe_load(f)['DRUG_DISEASE_MODEL_COLORS']

ROOT = Path(snakemake.input.aggregate)
scores = pd.read_pickle(ROOT / 'predictions_results_aggregated.pkl')
rename = {'module_based_archs4':'ARCHS4', 'module_based_recount2':'recount2', 'module_based_gtex':'GTEx', 'gene_based':'Gene-based'}
scores['model'] = scores.method.map(rename)
order = ['ARCHS4', 'recount2', 'GTEx', 'Gene-based']
assert set(scores.model) == set(order)
fig, ax = plt.subplots(figsize=(9, 4))
for model in order:
    ax.hist(scores.loc[scores.model == model, 'score'], bins=45, density=True, histtype='step', linewidth=2, label=model, color=colors[model])
ax.set(xlabel='Aggregated drug-disease score', ylabel='Density', title='Canonical model score distributions')
ax.legend(frameon=False)
plt.show()
scores.groupby('model', observed=True).score.describe()

In [ ]:
fig, ax_roc = plt.subplots(figsize=(7, 5.5))
for model in order:
    sub = scores.loc[scores.model == model]
    fpr, tpr, _ = roc_curve(sub.true_class, sub.score)
    ax_roc.plot(fpr, tpr, color=colors[model], linewidth=2, label=f'{model} (AUC={roc_auc_score(sub.true_class, sub.score):.3f})')
ax_roc.plot([0, 1], [0, 1], color='grey', linestyle='--', linewidth=1)
ax_roc.set(xlabel='False positive rate', ylabel='True positive rate', title='ROC curve', xlim=(0, 1), ylim=(0, 1))
ax_roc.legend(loc='lower right', fontsize=8, frameon=False)
fig.tight_layout()
plt.show()